# PyTorch, `torch.compile`, and Triton Softmax Benchmark

This notebook validates and benchmarks three row-wise softmax implementations on an NVIDIA GPU. Select **Runtime → Change runtime type → T4 GPU** (or another NVIDIA GPU) before running it.

The notebook treats compilation separately from steady-state execution and records the runtime environment with every result.

In [ ]:
# Replace YOUR-USERNAME after publishing the repository.
!git clone https://github.com/YOUR-USERNAME/gpu-pytorch-performance-lab.git
%cd gpu-pytorch-performance-lab
!pip install -q -e .

In [ ]:
import json
import platform
from pathlib import Path

import pandas as pd
import torch
import triton

from gpu_lab.benchmark import (
    benchmark_cuda,
    estimate_effective_bandwidth_gbps,
    validate_against_reference,
)
from gpu_lab.softmax import eager_softmax, naive_softmax, triton_softmax

assert torch.cuda.is_available(), "Select a GPU runtime before continuing."
device = torch.device("cuda")
environment = {
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "triton": triton.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "compute_capability": ".".join(map(str, torch.cuda.get_device_capability(0))),
}
environment

## Implementations

- **Naive PyTorch:** exposes separate reduction, exponential, and division operations.
- **PyTorch eager:** uses `torch.softmax`.
- **Compiled PyTorch:** compiles the naive function and may fuse operations.
- **Triton:** uses one fused program per matrix row.

In [ ]:
compiled_softmax = torch.compile(naive_softmax)
implementations = {
    "pytorch_eager": eager_softmax,
    "pytorch_naive": naive_softmax,
    "torch_compile": compiled_softmax,
    "triton": triton_softmax,
}

# Trigger compilation outside the measured region.
_compile_input = torch.randn(256, 1024, device=device, dtype=torch.float16)
compiled_softmax(_compile_input)
torch.cuda.synchronize()

In [ ]:
# Numerical validation must happen before performance comparison.
validation_input = torch.randn(128, 1024, device=device, dtype=torch.float16)
validation = {
    name: validate_against_reference(fn, validation_input)
    for name, fn in implementations.items()
}
pd.DataFrame(validation).T

In [ ]:
shapes = [(256, 256), (1024, 1024), (4096, 2048), (8192, 4096)]
dtype = torch.float16
rows = []

for shape in shapes:
    x = torch.randn(*shape, device=device, dtype=dtype)
    for name, fn in implementations.items():
        timing = benchmark_cuda(fn, x, warmup=25, repetitions=100)
        bandwidth = estimate_effective_bandwidth_gbps(
            shape, x.element_size(), timing["latency_median_ms"]
        )
        rows.append({
            "implementation": name,
            "rows": shape[0],
            "columns": shape[1],
            "dtype": str(dtype),
            **timing,
            "effective_bandwidth_gbps": bandwidth,
        })

results = pd.DataFrame(rows)
baseline = results.loc[results.implementation == "pytorch_eager", [
    "rows", "columns", "latency_median_ms"
]].rename(columns={"latency_median_ms": "eager_median_ms"})
results = results.merge(baseline, on=["rows", "columns"])
results["speedup_vs_eager"] = (
    results["eager_median_ms"] / results["latency_median_ms"]
)
results.sort_values(["rows", "columns", "latency_median_ms"])

In [ ]:
results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
results.to_csv(results_dir / "softmax_benchmark.csv", index=False)
(results_dir / "environment.json").write_text(
    json.dumps(environment, indent=2), encoding="utf-8"
)
print("Saved benchmark and environment metadata.")

## PyTorch Profiler trace

The trace below captures launches after warm-up. Download it from the Colab file browser and open it in Perfetto. A trace is environment-specific evidence and should be published together with the metadata.

In [ ]:
from torch.profiler import ProfilerActivity, profile, record_function

profile_input = torch.randn(2048, 2048, device=device, dtype=dtype)
for fn in implementations.values():
    fn(profile_input)
torch.cuda.synchronize()

with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
    for name, fn in implementations.items():
        with record_function(name):
            fn(profile_input)
    torch.cuda.synchronize()

trace_path = results_dir / "softmax_profiler.pt.trace.json"
prof.export_chrome_trace(str(trace_path))
print(prof.key_averages().table(sort_by="self_cuda_time_total", row_limit=15))
print(f"Trace saved to {trace_path}")

## Interpretation checklist

1. Which implementation has the lowest median latency for each shape?
2. Are p10 and p90 close enough to support a stable conclusion?
3. Does the trace show one fused kernel or several launches?
4. Is the estimated bandwidth plausible for the runtime GPU?
5. Does the conclusion change with FP32 or a different row width?
6. Which claims are supported only by this Colab environment, and which may generalize?